# Loading the dependencies

In [2]:
from datasets import load_dataset
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
import numpy as np

# Loading the data set and transforming

Load the dataset from the tensorflow keras and split into the training and test data

In [3]:
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
cnn_train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255.0
cnn_test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255.0

# CNN Model

Training for the CNN model using:


*   Optimizer = adam
*   loss function = sparse categorical crossentropy
*   Filter layers size = 32, 64
*   Hidden layers size = 64
*   Activation function = relu
*   Output layer size = 10
*   Epochs = 5

In [5]:
cnn_model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(10, activation='softmax')
])


cnn_model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])

cnn_model.fit(cnn_train_images, train_labels, epochs=5, batch_size=64, validation_split=0.1)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 17s 14ms/step - accuracy: 0.8838 - loss: 0.3810 - val_accuracy: 0.9838 - val_loss: 0.0596
Epoch 2/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9581 - loss: 0.1428 - val_accuracy: 0.9870 - val_loss: 0.0478
Epoch 3/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9683 - loss: 0.1068 - val_accuracy: 0.9880 - val_loss: 0.0402
Epoch 4/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - accuracy: 0.9737 - loss: 0.0873 - val_accuracy: 0.9900 - val_loss: 0.0394
Epoch 5/5
844/844 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.9779 - loss: 0.0754 - val_accuracy: 0.9907 - val_loss: 0.0331


# Model Attacks Evaluation

Defining the parameters and predict using the base model



In [6]:
epsilon = 0.3
alpha = 0.01
steps = 40
decay_factor = 1

Trained CNN model Accuracy = 98.98%

In [27]:
model_output = cnn_model.predict(cnn_test_images)
predictions = tf.argmax(model_output, axis=1)
correct_samples = (predictions == test_labels)
total_samples = tf.size(test_labels)
correct_predictions = tf.reduce_sum(tf.cast(correct_samples, tf.int32))
accuracy = correct_predictions / total_samples
print("Total Test Samples:", total_samples.numpy())
print("Model Correct Predictions:", correct_predictions.numpy())
print("Model Accuracy:", accuracy.numpy() * 100)

filtered_images = tf.convert_to_tensor(cnn_test_images[correct_samples])
filtered_labels = tf.convert_to_tensor(test_labels[correct_samples])

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
Total Test Samples: 10000
Model Correct Predictions: 9898
Model Accuracy: 98.98


## FGSM Algorithm

In [7]:
def fgsm_transform_image(images, labels):
  with tf.GradientTape() as tape:
    tape.watch(images)
    prediction = cnn_model(images)
    loss = tf.keras.losses.sparse_categorical_crossentropy(labels, prediction)
  gradient = tape.gradient(loss, images)
  signed_grad = tf.sign(gradient)
  perturbation = images + epsilon * signed_grad
  perturbed_images = tf.clip_by_value(images + perturbation, 0, 1)
  return perturbed_images

Total Images Perturbed = 9898

FGSM Successful Attacks = 3543

Successful Attack Rate = 35.80%

In [25]:
fgsm_perturbed_images = fgsm_transform_image(filtered_images, filtered_labels)
fgsm_predictions = tf.argmax(cnn_model.predict(fgsm_perturbed_images), axis=1)
fgsm_successful_attacks = tf.reduce_sum(tf.cast(fgsm_predictions != tf.cast(filtered_labels, tf.int64), tf.int64))

print("FGSM Successful Attacks:", fgsm_successful_attacks.numpy())
print("FGSM Successful Attack rate:", (fgsm_successful_attacks.numpy() / correct_predictions.numpy()) * 100)

310/310 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
FGSM Successful Attacks: 3543
FGSM Successful Attack rate: 35.795110123257224


## IFGSM/PGD Algorithm

In [15]:
def pgd_transform_image(images, labels):
  perbutation = tf.random.uniform(tf.shape(images), minval=-epsilon, maxval=epsilon)
  perturbed_images = tf.clip_by_value(images + perbutation, 0, 1)

  for _ in range(steps):
    with tf.GradientTape() as tape:
      tape.watch(perturbed_images)
      prediction = cnn_model(perturbed_images)
      loss = tf.keras.losses.sparse_categorical_crossentropy(labels, prediction)

    gradient = tape.gradient(loss, perturbed_images)
    signed_grad = tf.sign(gradient)
    perturbation = alpha * signed_grad

    int_images = perturbed_images + perturbation

    perturbed_images = tf.clip_by_value(int_images - images, -epsilon, epsilon)
    perturbed_images = tf.clip_by_value(images + perturbed_images, 0, 1)

  return perturbed_images

Total Images Perturbed = 9898

PGD Successful Attacks = 9898

Successful Attack Rate = 100%

In [28]:
pgd_perturbed_images = pgd_transform_image(filtered_images, filtered_labels)
pgd_predictions = tf.argmax(cnn_model.predict(pgd_perturbed_images), axis=1)
pgd_successful_attacks = tf.reduce_sum(tf.cast(pgd_predictions != tf.cast(filtered_labels, tf.int64), tf.int64))

print("PGD Successful Attacks:", pgd_successful_attacks.numpy())
print("PGD Successful Attack rate:", (pgd_successful_attacks.numpy() / correct_predictions.numpy()) * 100)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
PGD Successful Attacks: 9898
PGD Successful Attack rate: 100.0


## MIFGSM Algorithm

In [19]:
def mifgsm_transform_image(images, labels):
  perturbed_images = images
  momentum = tf.zeros_like(images)

  for _ in range(steps):
    with tf.GradientTape() as tape:
      tape.watch(perturbed_images)
      prediction = cnn_model(perturbed_images)
      loss = tf.keras.losses.sparse_categorical_crossentropy(labels, prediction)

    gradient = tape.gradient(loss, perturbed_images)

    grad_norm = tf.reduce_sum(tf.abs(gradient), axis=[1,2,3], keepdims=True)
    normalized_gradient = gradient / (grad_norm + 1e-8)

    momentum = decay_factor * momentum + normalized_gradient
    perturbation = alpha * tf.sign(momentum)

    int_images = perturbed_images + perturbation

    perturbed_images = tf.clip_by_value(int_images - images, -epsilon, epsilon)
    perturbed_images = tf.clip_by_value(images + perturbed_images, 0, 1)

  return perturbed_images

Total Images Perturbed = 9898

MIFGSM Successful Attacks = 9878

Successful Attack Rate = 99.80%

In [29]:
mifgsm_perturbed_images = mifgsm_transform_image(filtered_images, filtered_labels)
mifgsm_predictions = tf.argmax(cnn_model.predict(mifgsm_perturbed_images), axis=1)
mifgsm_successful_attacks = tf.reduce_sum(tf.cast(mifgsm_predictions != tf.cast(filtered_labels, tf.int64), tf.int64))

print("MIFGSM Successful Attacks:", mifgsm_successful_attacks.numpy())
print("MIFGSM Successful Attack rate:", (mifgsm_successful_attacks.numpy() / correct_predictions.numpy()) * 100)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step
MIFGSM Successful Attacks: 9878
MIFGSM Successful Attack rate: 99.79793897757122
